In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils import get_model_metrics
from matplotlib.colors import Normalize
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from atow_estimation.paths import PROCESSED_DATA_DIR
import optuna
from catboost import CatBoostRegressor, metrics
from utils import preprocess_catboost
from atow_estimation.config import N_TRIALS, get_catboost_params, CATEGORICAL_VARIABLES
from catboost import CatBoostRegressor, Pool
import pickle

In [130]:
data_m1_M = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'processed_data_m1_M.csv'))
data_m2_M = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'processed_data_m2_M.csv'))
data_H = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'processed_data_H.csv'))
data_M = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'processed_data_M.csv'))

In [11]:
def print_study_results(study):
    """Prints the best trial information from an Optuna study."""
    print(f"Best trial number: {study.best_trial.number}")
    print(f"Best RMSE: {study.best_trial.value:.4f}")
    print("Best hyperparameters:")
    for k, v in study.best_trial.params.items():
        print(f"  {k}: {v}")


def print_model_metrics(r2, mae, rmse, mape):
    """Prints the evaluation metrics of a trained model."""
    print(f"  R^2 Score: {r2:.4f}")
    print(f"  Mean Absolute Error: {mae:.2f}")
    print(f"  Root Mean Squared Error: {rmse:.2f}")
    print(f"  Mean Absolute Percentage Error: {mape:.4f}")

def objective_catboost(trial, train_pool, test_pool):
    params = get_catboost_params(trial)
    model = CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=params["od_wait"], verbose=1000)
    best_rmse_on_val = model.get_best_score()["validation"]["RMSE"]
    trial.set_user_attr("best_iteration", model.get_best_iteration())
    return best_rmse_on_val

def run_hyperparameter_tuning(objective_fn, n_trials, *args):
    """Runs hyperparameter tuning using Optuna."""
    study = optuna.create_study(direction="minimize")
    study.optimize(
        lambda trial: objective_fn(trial, *args),
        n_trials=n_trials,
        timeout=21600,  # 6 hours
    )
    print_study_results(study)
    return study

# 4 models

### 1. Medium_wake and national flights


In [12]:
X_train, X_test, y_train, y_test, train_pool, test_pool = preprocess_catboost(data_m1_M)
tuning_args = (train_pool, test_pool)
print(f"Starting hyperparameter tuning for...")
study = run_hyperparameter_tuning(objective_catboost, N_TRIALS, *tuning_args)
print("Hyperparameter tuning finished.")

[I 2025-06-05 11:59:04,912] A new study created in memory with name: no-name-a104faeb-ef5f-4159-9070-9c7cc7bfcca6


Starting hyperparameter tuning for...
0:	learn: 7187.5363017	test: 7133.4081970	best: 7133.4081970 (0)	total: 60.2ms	remaining: 3m
1000:	learn: 632.2269711	test: 1438.7534956	best: 1438.6351590 (997)	total: 1m 5s	remaining: 2m 10s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1414.911917
bestIteration = 1868

Shrink model to first 1869 iterations.


[I 2025-06-05 12:01:13,158] Trial 0 finished with value: 1414.9119172004832 and parameters: {'learning_rate': 0.053274291841452076, 'depth': 8, 'random_strength': 35.107790078079304, 'colsample_bylevel': 0.10475873751394359, 'min_data_in_leaf': 33, 'leaf_estimation_iterations': 10}. Best is trial 0 with value: 1414.9119172004832.


0:	learn: 7385.0274196	test: 7324.9161618	best: 7324.9161618 (0)	total: 41.5ms	remaining: 2m 4s
1000:	learn: 2650.1952861	test: 2597.0588083	best: 2597.0588083 (1000)	total: 39.5s	remaining: 1m 18s
2000:	learn: 2106.6155838	test: 2112.2390769	best: 2112.2390769 (2000)	total: 1m 18s	remaining: 39s


[I 2025-06-05 12:03:10,411] Trial 1 finished with value: 1822.7834673266532 and parameters: {'learning_rate': 0.006652641333457126, 'depth': 4, 'random_strength': 11.935772940675443, 'colsample_bylevel': 0.15211539728379797, 'min_data_in_leaf': 24, 'leaf_estimation_iterations': 14}. Best is trial 0 with value: 1414.9119172004832.


...

2999:	learn: 1993.5399240	test: 2028.6560680	best: 2028.6560680 (2999)	total: 1m 40s	remaining: 0us

bestTest = 2028.656068
bestIteration = 2999

0:	learn: 7398.6555558	test: 7338.1097020	best: 7338.1097020 (0)	total: 64.1ms	remaining: 3m 12s
1000:	learn: 2934.9744492	test: 2880.7406377	best: 2880.7406377 (1000)	total: 56.2s	remaining: 1m 52s
2000:	learn: 2405.2651175	test: 2383.8410425	best: 2383.8410425 (2000)	total: 1m 53s	remaining: 56.8s


[I 2025-06-05 12:27:50,278] Trial 8 finished with value: 2168.409017855931 and parameters: {'learning_rate': 0.003220306627069509, 'depth': 6, 'random_strength': 12.34198304961556, 'colsample_bylevel': 0.6284364001274351, 'min_data_in_leaf': 24, 'leaf_estimation_iterations': 1}. Best is trial 6 with value: 1337.5397183496034.


2999:	learn: 2151.2037527	test: 2168.4090179	best: 2168.4090179 (2999)	total: 2m 52s	remaining: 0us

bestTest = 2168.409018
bestIteration = 2999

0:	learn: 7350.8648788	test: 7293.4668438	best: 7293.4668438 (0)	total: 195ms	remaining: 9m 43s
1000:	learn: 1348.1521627	test: 1752.6252831	best: 1752.6252831 (1000)	total: 2m 9s	remaining: 4m 18s
2000:	learn: 639.9575860	test: 1464.5012230	best: 1464.4765796 (1999)	total: 4m 36s	remaining: 2m 17s
2999:	learn: 376.3598070	test: 1436.9077359	best: 1436.9077359 (2999)	total: 7m 8s	remaining: 0us

bestTest = 1436.907736
bestIteration = 2999



[I 2025-06-05 12:35:00,301] Trial 9 finished with value: 1436.907735930441 and parameters: {'learning_rate': 0.013522399340988684, 'depth': 10, 'random_strength': 35.5023875011122, 'colsample_bylevel': 0.15241089444987344, 'min_data_in_leaf': 9, 'leaf_estimation_iterations': 9}. Best is trial 6 with value: 1337.5397183496034.


Best trial number: 6
Best RMSE: 1337.5397
Best hyperparameters:
  learning_rate: 0.028676272868598352
  depth: 5
  random_strength: 16.32020678101461
  colsample_bylevel: 0.8465919386377148
  min_data_in_leaf: 18
  leaf_estimation_iterations: 2
Hyperparameter tuning finished.


In [77]:
new_categorical_variables = [
    "RECATwake",
    "aircraftType",
    "airlineCode",
]
columns_to_remove = [
    'CLIMB_vel_mod_median',
    'CRUISE_vel_z_variance',
    'CRUISE_modo_c_variance',
    'CLIMB_vel_mod_variance_60_90fl',
    'CLIMB_vel_z_variance_30_90fl',
    'CLIMB_vel_mod_mean_60_90fl',
    'flightType',
    'CRUISE_vel_mod_variance',
    'CLIMB_vel_mod_variance_30_90fl',
    'CLIMB_vel_z_variance_30_60fl',
    'CRUISE_delta_modo_c',
    'routeType',
    'DESCENT_vel_mod_median',
    'DESCENT_delta_modo_c',
    'DESCENT_duration',
    'DESCENT_modo_c_variance',
    'DESCENT_vel_z_variance',
    'DESCENT_vel_mod_variance',
    'DESCENT_distance',
    'DESCENT_vel_z_median',
    'DESCENT_modo_c_median',
    'CRUISE_vel_z_median',
    'numberOfEngines',
'CLIMB_vel_z_variance_0_90fl',
'CLIMB_vel_mod_variance_0_30fl',
'CLIMB_modo_c_median',
'ALDT_track_hour',
'CLIMB_vel_z_variance_0_60fl',
'CLIMB_vel_z_mean_30_60fl',
'CLIMB_vel_mod_variance_30_60fl',
'CRUISE_vel_mod_median',    
    ]
new_cols = [col for col in X_train.columns if col not in columns_to_remove]
# Feature selection
X_train_new = X_train[new_cols]
X_test_new = X_test[new_cols]

In [78]:
best_params = study.best_trial.params.copy()
final_model = None
# iterations = study.best_trial.user_attrs.get("best_iteration", 1000)
cat_params = {
    **best_params,
    "iterations": 10000,
    "random_seed": 42,
    "loss_function": "RMSE",
    "task_type": "CPU",
    "allow_writing_files": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,    
}

final_model = CatBoostRegressor(**cat_params)

# train_pool = Pool(X_train, y_train, cat_features=CATEGORICAL_VARIABLES)
# test_pool = Pool(X_test, y_test, cat_features=CATEGORICAL_VARIABLES)
train_pool = Pool(X_train_new, y_train, cat_features=new_categorical_variables)
test_pool = Pool(X_test_new, y_test, cat_features=new_categorical_variables)
final_model.fit(
    train_pool,
    eval_set=test_pool,
    verbose=1000,
)

0:	learn: 7274.3008628	test: 7216.2682965	best: 7216.2682965 (0)	total: 37.2ms	remaining: 6m 12s
1000:	learn: 1303.2247535	test: 1499.0322868	best: 1499.0322868 (1000)	total: 46.2s	remaining: 6m 55s
2000:	learn: 1030.5250238	test: 1381.2258275	best: 1381.2229282 (1996)	total: 1m 25s	remaining: 5m 43s
3000:	learn: 887.5055648	test: 1351.4835788	best: 1351.4835788 (3000)	total: 2m 4s	remaining: 4m 49s
4000:	learn: 785.2067693	test: 1338.2259644	best: 1338.0828111 (3999)	total: 2m 43s	remaining: 4m 5s
5000:	learn: 702.8738901	test: 1330.0345801	best: 1330.0153525 (4999)	total: 3m 27s	remaining: 3m 27s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1326.198347
bestIteration = 5388

Shrink model to first 5389 iterations.


In [79]:
y_pred = final_model.predict(X_test_new)
r2, mae, rmse, mape = get_model_metrics(y_pred, X_test_new, y_test)
print_model_metrics(r2, mae, rmse, mape)

  R^2 Score: 0.9675
  Mean Absolute Error: 921.65
  Root Mean Squared Error: 1326.20
  Mean Absolute Percentage Error: 0.0145


In [19]:
with open('CatBoost_m1_M.pkl', 'wb') as file:  
    pickle.dump(final_model, file)

In [81]:
feature_importances = final_model.get_feature_importance(train_pool)
feature_names = X_train_new.columns
for score, name in sorted(zip(feature_importances, feature_names), reverse=True):
    print('{}: {}'.format(name, score))
print(len(feature_importances))

mfc: 11.464186983836312
mtow_rates: 8.735370805911028
mtow_openap: 7.545259327849109
RECATwake: 5.945590145193855
Payload: 5.478819761554956
seats: 5.439965651372924
oew: 5.246573697670885
CLIMB_vel_z_median: 4.1653105891606925
aircraftType: 3.5581463595117837
mlw: 3.2875633669229565
CRUISE_modo_c_median: 3.0102486969270768
airlineCode: 2.406833101891917
cruiseLevel: 2.222223527014048
CLIMB_duration: 2.057215258395531
CRUISE_distance: 1.909283120653484
ADESLong: 1.59525347864164
total_distance: 1.581770990242582
ADESLat: 1.37634139929579
CLIMB_delta_modo_c: 1.3488376238563553
cruiseSpeed: 1.3375081787680456
CLIMB_modo_c_variance: 1.2956642774815452
CLIMB_distance: 1.1926313278604164
CLIMB_vel_z_mean_0_30fl: 1.1859869817373503
ADEPLong: 1.180695750483627
CLIMB_vel_z_mean_0_90fl: 1.1310022051293611
CLIMB_vel_mod_mean_30_90fl: 0.9964752392835869
CLIMB_vel_z_variance: 0.9659383640250448
CLIMB_vel_z_mean_60_90fl: 0.9635177693500836
ADEPLat: 0.9545343287401924
CRUISE_duration: 0.872497696394

In [76]:
for score, name in sorted(zip(feature_importances, feature_names), reverse=True):
    if score < 0.4:
        print("'{}',".format(name))

'CLIMB_vel_z_variance_0_90fl',
'CLIMB_vel_mod_variance_0_30fl',
'CLIMB_modo_c_median',
'ALDT_track_hour',
'CLIMB_vel_z_variance_0_60fl',
'CLIMB_vel_z_mean_30_60fl',
'CLIMB_vel_mod_variance_30_60fl',
'CRUISE_vel_mod_median',


### 2. Medium_wake and international flights


In [84]:
X_train, X_test, y_train, y_test, train_pool, test_pool = preprocess_catboost(data_m2_M)
tuning_args = (train_pool, test_pool)
print(f"Starting hyperparameter tuning for...")
study = run_hyperparameter_tuning(objective_catboost, N_TRIALS, *tuning_args)
print("Hyperparameter tuning finished.")

[I 2025-06-05 15:32:01,918] A new study created in memory with name: no-name-d61ac3ae-c820-479b-b25b-0b4e38f628f8


Starting hyperparameter tuning for...
0:	learn: 8124.2688058	test: 8148.6282484	best: 8148.6282484 (0)	total: 101ms	remaining: 5m 4s
1000:	learn: 4242.5343638	test: 4271.4962516	best: 4271.4962516 (1000)	total: 1m 41s	remaining: 3m 23s
2000:	learn: 3219.3629427	test: 3255.3588158	best: 3255.3588158 (2000)	total: 3m 16s	remaining: 1m 38s
2999:	learn: 2858.5611274	test: 2901.0774184	best: 2901.0774184 (2999)	total: 4m 44s	remaining: 0us

bestTest = 2901.077418
bestIteration = 2999



[I 2025-06-05 15:36:48,014] Trial 0 finished with value: 2901.0774183789345 and parameters: {'learning_rate': 0.0012745735066458867, 'depth': 9, 'random_strength': 37.4645921349969, 'colsample_bylevel': 0.29027753804714784, 'min_data_in_leaf': 26, 'leaf_estimation_iterations': 2}. Best is trial 0 with value: 2901.0774183789345.


0:	learn: 8096.4343061	test: 8121.0232098	best: 8121.0232098 (0)	total: 62.8ms	remaining: 3m 8s
1000:	learn: 2697.5825401	test: 2722.7451474	best: 2722.7451474 (1000)	total: 1m 33s	remaining: 3m 7s
2000:	learn: 2143.4352500	test: 2230.0278592	best: 2230.0278592 (2000)	total: 3m 2s	remaining: 1m 31s
2999:	learn: 1740.1158572	test: 1923.8669529	best: 1923.8669529 (2999)	total: 4m 30s	remaining: 0us

bestTest = 1923.866953
bestIteration = 2999



[I 2025-06-05 15:41:19,497] Trial 1 finished with value: 1923.8669528566013 and parameters: {'learning_rate': 0.0066168198912326705, 'depth': 7, 'random_strength': 39.75838349207559, 'colsample_bylevel': 0.39645556771658647, 'min_data_in_leaf': 14, 'leaf_estimation_iterations': 7}. Best is trial 1 with value: 1923.8669528566013.


...

0:	learn: 8114.5220405	test: 8138.6594979	best: 8138.6594979 (0)	total: 42.8ms	remaining: 2m 8s
1000:	learn: 3474.1618329	test: 3469.6268545	best: 3469.6268545 (1000)	total: 46.5s	remaining: 1m 32s
2000:	learn: 2906.3284709	test: 2893.0681077	best: 2893.0681077 (2000)	total: 1m 34s	remaining: 47.2s


[I 2025-06-05 16:08:37,205] Trial 8 finished with value: 2718.35851610003 and parameters: {'learning_rate': 0.0027720471657588395, 'depth': 4, 'random_strength': 20.270932710090538, 'colsample_bylevel': 0.12949995521236043, 'min_data_in_leaf': 17, 'leaf_estimation_iterations': 13}. Best is trial 5 with value: 1722.4279331269047.


2999:	learn: 2726.8352213	test: 2718.3585161	best: 2718.3585161 (2999)	total: 2m 17s	remaining: 0us

bestTest = 2718.358516
bestIteration = 2999

0:	learn: 8015.2499417	test: 8038.7285668	best: 8038.7285668 (0)	total: 49.1ms	remaining: 2m 27s
1000:	learn: 1916.0520670	test: 2051.7241155	best: 2051.7241155 (1000)	total: 37.2s	remaining: 1m 14s
2000:	learn: 1645.3894795	test: 1901.2794657	best: 1901.2794657 (2000)	total: 1m 14s	remaining: 37s


[I 2025-06-05 16:10:30,034] Trial 9 finished with value: 1837.7571578329419 and parameters: {'learning_rate': 0.024368248493332804, 'depth': 6, 'random_strength': 11.363403067891765, 'colsample_bylevel': 0.05649691616792504, 'min_data_in_leaf': 25, 'leaf_estimation_iterations': 11}. Best is trial 5 with value: 1722.4279331269047.


2999:	learn: 1492.7521498	test: 1837.7571578	best: 1837.7571578 (2999)	total: 1m 51s	remaining: 0us

bestTest = 1837.757158
bestIteration = 2999

Best trial number: 5
Best RMSE: 1722.4279
Best hyperparameters:
  learning_rate: 0.03173811672075018
  depth: 9
  random_strength: 10.726397400971534
  colsample_bylevel: 0.27940971637032447
  min_data_in_leaf: 48
  leaf_estimation_iterations: 7
Hyperparameter tuning finished.


In [101]:
new_categorical_variables = [
    "routeType",
    "RECATwake",
    "aircraftType",
    "airlineCode",
]

columns_to_remove = [
'DESCENT_modo_c_median',
'CRUISE_modo_c_variance',
'CRUISE_delta_modo_c',
'DESCENT_duration',
'CLIMB_vel_z_median',
'flightType',
'CRUISE_vel_z_variance',
'CLIMB_duration',
'CLIMB_vel_z_variance',
'CLIMB_distance',
'CLIMB_modo_c_variance',
'CLIMB_delta_modo_c',
'CLIMB_vel_mod_variance',
'CLIMB_vel_mod_median',
'CLIMB_modo_c_median',
'CRUISE_vel_z_median',
'numberOfEngines',
    ]
new_cols = [col for col in X_train.columns if col not in columns_to_remove]
# Feature selection
X_train_new = X_train[new_cols]
X_test_new = X_test[new_cols]

In [102]:
best_params = study.best_trial.params.copy()
final_model = None
# iterations = study.best_trial.user_attrs.get("best_iteration", 1000)
cat_params = {
    **best_params,
    "iterations": 10000,
    "random_seed": 42,
    "loss_function": "RMSE",
    "task_type": "CPU",
    "allow_writing_files": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,    
}

final_model = CatBoostRegressor(**cat_params)

# train_pool = Pool(X_train, y_train, cat_features=CATEGORICAL_VARIABLES)
# test_pool = Pool(X_test, y_test, cat_features=CATEGORICAL_VARIABLES)
train_pool = Pool(X_train_new, y_train, cat_features=new_categorical_variables)
test_pool = Pool(X_test_new, y_test, cat_features=new_categorical_variables)
final_model.fit(
    train_pool,
    eval_set=test_pool,
    verbose=1000,
)

0:	learn: 7930.7429823	test: 7952.6507853	best: 7952.6507853 (0)	total: 71.4ms	remaining: 11m 53s
1000:	learn: 1194.7481634	test: 1777.3032969	best: 1777.3032969 (1000)	total: 1m 29s	remaining: 13m 23s
2000:	learn: 805.8990542	test: 1739.5660600	best: 1739.5207274 (1996)	total: 3m 10s	remaining: 12m 40s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1730.565616
bestIteration = 2760

Shrink model to first 2761 iterations.


In [103]:
y_pred = final_model.predict(X_test_new)
r2, mae, rmse, mape =  get_model_metrics(y_pred, X_test_new, y_test)    
print(f"Mean Absolute Error: {mae:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"R^2 Score: {r2:.4f}")

Mean Absolute Error: 1145.34
Root Mean Squared Error: 1730.57
R^2 Score: 0.9550


In [87]:
with open('CatBoost_m2_M.pkl', 'wb') as file:  
    pickle.dump(final_model, file)

In [104]:
feature_importances = final_model.get_feature_importance(train_pool)
feature_names = X_train_new.columns
for score, name in sorted(zip(feature_importances, feature_names), reverse=True):
    print('{}: {}'.format(name, score))

aircraftType: 9.3331003063524
oew: 8.893967889905584
mtow_rates: 7.377201475448827
cruiseLevel: 5.994906113884488
CRUISE_modo_c_median: 5.271712842624002
mfc: 5.2495515804581
mtow_openap: 4.806843178031571
mlw: 4.737919684925887
airlineCode: 4.277411424137294
cruiseSpeed: 3.4421575964108464
total_distance: 3.1065170339617896
CRUISE_duration: 3.0036925273200423
ADESLong: 2.716921396070704
seats: 2.5241568595562667
ALDT_track_day: 2.3752108775197898
CRUISE_distance: 2.3207644765803432
ATOT_track_day: 2.285613065245518
RECATwake: 2.2454543944668135
Payload: 1.8929401312709437
flight_duration: 1.6844758091148755
ADEPLong: 1.595931953797129
ADEPLat: 1.5652833153006425
DESCENT_delta_modo_c: 1.5482143279370149
ADESLat: 1.5396248941946151
DESCENT_modo_c_variance: 1.1333582423637443
ATOT_track_hour: 1.1039919247240928
CRUISE_vel_mod_median: 1.0915661270757138
CRUISE_vel_mod_variance: 1.0531588057558388
routeType: 0.9742229466080192
DESCENT_vel_z_variance: 0.9579114594634611
ALDT_track_hour: 0.8

In [105]:
for score, name in sorted(zip(feature_importances, feature_names), reverse=True):
    if score < 0.4:
        print("'{}',".format(name))

In [109]:
feature_names

Index(['ADEPLat', 'ADEPLong', 'ADESLat', 'ADESLong', 'routeType',
       'aircraftType', 'airlineCode', 'RECATwake', 'cruiseSpeed',
       'cruiseLevel', 'ATOT_track_hour', 'ATOT_track_day', 'ALDT_track_hour',
       'ALDT_track_day', 'flight_duration', 'CRUISE_duration',
       'total_distance', 'CRUISE_distance', 'DESCENT_distance',
       'CRUISE_modo_c_median', 'CRUISE_vel_mod_median',
       'CRUISE_vel_mod_variance', 'DESCENT_modo_c_variance',
       'DESCENT_vel_mod_median', 'DESCENT_vel_mod_variance',
       'DESCENT_vel_z_median', 'DESCENT_vel_z_variance',
       'DESCENT_delta_modo_c', 'mtow_openap', 'mlw', 'mfc', 'oew',
       'mtow_rates', 'seats', 'Payload'],
      dtype='object')

### 3. All flights with Heavy_wake

In [110]:
X_train, X_test, y_train, y_test, train_pool, test_pool = preprocess_catboost(data_H)
tuning_args = (train_pool, test_pool)
print(f"Starting hyperparameter tuning for...")
study = run_hyperparameter_tuning(objective_catboost, N_TRIALS, *tuning_args)
print("Hyperparameter tuning finished.")

[I 2025-06-06 10:25:36,336] A new study created in memory with name: no-name-6ac66d16-769a-4b01-a1a8-8cdc94ec06dd


Starting hyperparameter tuning for...
0:	learn: 25401.2374666	test: 24982.9818422	best: 24982.9818422 (0)	total: 26.4ms	remaining: 1m 19s
1000:	learn: 3981.6122027	test: 5449.5681213	best: 5447.4653135 (980)	total: 25.9s	remaining: 51.7s


[I 2025-06-06 10:26:09,785] Trial 0 finished with value: 5392.269247718237 and parameters: {'learning_rate': 0.09481127224251423, 'depth': 3, 'random_strength': 38.53683997491159, 'colsample_bylevel': 0.39624235108244604, 'min_data_in_leaf': 47, 'leaf_estimation_iterations': 13}. Best is trial 0 with value: 5392.269247718237.


Stopped by overfitting detector  (50 iterations wait)

bestTest = 5392.269248
bestIteration = 1266

Shrink model to first 1267 iterations.
0:	learn: 25103.6789674	test: 24696.7667745	best: 24696.7667745 (0)	total: 56.2ms	remaining: 2m 48s
1000:	learn: 4581.0705073	test: 5684.4616432	best: 5684.4616432 (1000)	total: 35.8s	remaining: 1m 11s
2000:	learn: 3115.8015810	test: 5381.2587837	best: 5381.2405578 (1999)	total: 1m 14s	remaining: 37.1s


[I 2025-06-06 10:28:02,132] Trial 1 finished with value: 5297.024756157547 and parameters: {'learning_rate': 0.017000254735932783, 'depth': 7, 'random_strength': 28.822295645826497, 'colsample_bylevel': 0.3006356609357159, 'min_data_in_leaf': 41, 'leaf_estimation_iterations': 4}. Best is trial 1 with value: 5297.024756157547.


...

2999:	learn: 2828.8698058	test: 5214.2740792	best: 5214.1230613 (2997)	total: 2m 3s	remaining: 0us

bestTest = 5214.123061
bestIteration = 2997

Shrink model to first 2998 iterations.
0:	learn: 25419.1686766	test: 25010.3352444	best: 25010.3352444 (0)	total: 210ms	remaining: 10m 28s
1000:	learn: 9751.4624994	test: 9729.2535499	best: 9729.2535499 (1000)	total: 1m 50s	remaining: 3m 41s
2000:	learn: 6914.1277788	test: 7244.3553134	best: 7244.3553134 (2000)	total: 3m 35s	remaining: 1m 47s
2999:	learn: 6118.8648536	test: 6700.9248450	best: 6700.9248450 (2999)	total: 5m 25s	remaining: 0us

bestTest = 6700.924845
bestIteration = 2999



[I 2025-06-06 10:42:27,408] Trial 8 finished with value: 6700.924844968656 and parameters: {'learning_rate': 0.001865560333132554, 'depth': 10, 'random_strength': 41.13847674200794, 'colsample_bylevel': 0.5446935211139472, 'min_data_in_leaf': 32, 'leaf_estimation_iterations': 5}. Best is trial 7 with value: 5214.123061345305.


0:	learn: 25425.6960278	test: 25017.0498801	best: 25017.0498801 (0)	total: 61.7ms	remaining: 3m 4s
1000:	learn: 11911.7052369	test: 11611.4426667	best: 11611.4426667 (1000)	total: 1m	remaining: 2m
2000:	learn: 8216.0656843	test: 8126.0086938	best: 8126.0086938 (2000)	total: 1m 58s	remaining: 59.4s
2999:	learn: 7047.6935501	test: 7159.8917855	best: 7159.8917855 (2999)	total: 3m	remaining: 0us

bestTest = 7159.891786
bestIteration = 2999



[I 2025-06-06 10:45:28,848] Trial 9 finished with value: 7159.891785545538 and parameters: {'learning_rate': 0.001510014512199736, 'depth': 8, 'random_strength': 34.56244738786253, 'colsample_bylevel': 0.537304594961219, 'min_data_in_leaf': 19, 'leaf_estimation_iterations': 3}. Best is trial 7 with value: 5214.123061345305.


Best trial number: 7
Best RMSE: 5214.1231
Best hyperparameters:
  learning_rate: 0.023250329590630295
  depth: 5
  random_strength: 11.004473566061437
  colsample_bylevel: 0.8146433469959493
  min_data_in_leaf: 10
  leaf_estimation_iterations: 15
Hyperparameter tuning finished.


In [118]:
new_categorical_variables = [
    "routeType",
    "RECATwake",
    "aircraftType",
    "airlineCode",
]

columns_to_remove = [
'DESCENT_vel_mod_variance',
'CRUISE_vel_mod_variance',
'flightType',
'DESCENT_vel_z_variance',
'CLIMB_vel_z_variance',
'CRUISE_modo_c_variance',
'CRUISE_vel_z_variance',
'CLIMB_vel_mod_variance',
'CLIMB_delta_modo_c',
'CLIMB_distance',
'DESCENT_distance',
'DESCENT_delta_modo_c',
'DESCENT_duration',
'DESCENT_modo_c_median',
'CLIMB_duration',
'CLIMB_modo_c_median',
'CRUISE_delta_modo_c',
'DESCENT_modo_c_variance',
'DESCENT_vel_mod_median',
'CLIMB_vel_mod_median',
'DESCENT_vel_z_median',
'numberOfEngines',
'CRUISE_vel_z_median',
    ]
new_cols = [col for col in X_train.columns if col not in columns_to_remove]
# Feature selection
X_train_new = X_train[new_cols]
X_test_new = X_test[new_cols]

In [119]:
best_params = study.best_trial.params.copy()
final_model = None
# iterations = study.best_trial.user_attrs.get("best_iteration", 1000)
cat_params = {
    **best_params,
    "iterations": 10000,
    "random_seed": 42,
    "loss_function": "RMSE",
    "task_type": "CPU",
    "allow_writing_files": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,    
}

final_model = CatBoostRegressor(**cat_params)

# train_pool = Pool(X_train, y_train, cat_features=CATEGORICAL_VARIABLES)
# test_pool = Pool(X_test, y_test, cat_features=CATEGORICAL_VARIABLES)
train_pool = Pool(X_train_new, y_train, cat_features=new_categorical_variables)
test_pool = Pool(X_test_new, y_test, cat_features=new_categorical_variables)
final_model.fit(
    train_pool,
    eval_set=test_pool,
    verbose=1000,
)

0:	learn: 24994.9199707	test: 24587.7829511	best: 24587.7829511 (0)	total: 29.2ms	remaining: 4m 51s
1000:	learn: 4613.1805256	test: 5581.1451285	best: 5581.1451285 (1000)	total: 41.7s	remaining: 6m 15s
2000:	learn: 3681.8957100	test: 5295.0540685	best: 5295.0093093 (1999)	total: 1m 19s	remaining: 5m 17s
3000:	learn: 3127.8729961	test: 5217.4566506	best: 5217.0952541 (2992)	total: 1m 54s	remaining: 4m 26s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 5215.609894
bestIteration = 3035

Shrink model to first 3036 iterations.


In [120]:
y_pred = final_model.predict(X_test_new)
r2, mae, rmse, mape =  get_model_metrics(y_pred, X_test_new, y_test)    
print(f"Mean Absolute Error: {mae:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"R^2 Score: {r2:.4f}")

Mean Absolute Error: 3886.11
Root Mean Squared Error: 5215.61
R^2 Score: 0.9566


In [114]:
with open('CatBoost_H.pkl', 'wb') as file:  
    pickle.dump(final_model, file)

In [122]:
feature_importances = final_model.get_feature_importance(train_pool)
feature_names = X_train_new.columns
for score, name in sorted(zip(feature_importances, feature_names), reverse=True):
    print('{}: {}'.format(name, score))

airlineCode: 11.048855096642708
mlw: 10.282319327015088
mfc: 9.504962639072405
aircraftType: 9.030750507471225
seats: 8.93243554435024
mtow_openap: 8.910777030668134
oew: 6.778002333841143
mtow_rates: 6.527200541853112
total_distance: 4.334596059661864
Payload: 3.6082679407287657
CRUISE_distance: 1.7487849388308374
ALDT_track_day: 1.7240109972578297
CRUISE_duration: 1.6098780625960734
cruiseLevel: 1.4578283744230793
ATOT_track_day: 1.4302409711813164
CRUISE_modo_c_median: 1.3431051090256996
cruiseSpeed: 1.3414366062965886
flight_duration: 1.2850161220200207
routeType: 1.1779123898862482
CLIMB_vel_z_median: 1.0060933678203385
ALDT_track_hour: 1.0052705929113315
ADEPLat: 0.960575236880136
ADESLat: 0.8915899274973024
ADESLong: 0.8608068137277957
ADEPLong: 0.7277328826626187
CRUISE_vel_mod_median: 0.6727080481977581
CLIMB_modo_c_variance: 0.6566988963017976
ATOT_track_hour: 0.5750790557535117
RECATwake: 0.5670645854250895


In [123]:
for score, name in sorted(zip(feature_importances, feature_names), reverse=True):
    if score < 0.4:
        print("'{}',".format(name))

In [126]:
feature_names

Index(['ADEPLat', 'ADEPLong', 'ADESLat', 'ADESLong', 'routeType',
       'aircraftType', 'airlineCode', 'RECATwake', 'cruiseSpeed',
       'cruiseLevel', 'ATOT_track_hour', 'ATOT_track_day', 'ALDT_track_hour',
       'ALDT_track_day', 'flight_duration', 'CRUISE_duration',
       'total_distance', 'CRUISE_distance', 'CLIMB_modo_c_variance',
       'CLIMB_vel_z_median', 'CRUISE_modo_c_median', 'CRUISE_vel_mod_median',
       'mtow_openap', 'mlw', 'mfc', 'oew', 'mtow_rates', 'seats', 'Payload'],
      dtype='object')

### 4. All flights with Medium_wake

In [131]:
X_train, X_test, y_train, y_test, train_pool, test_pool = preprocess_catboost(data_M)
tuning_args = (train_pool, test_pool)
print(f"Starting hyperparameter tuning for...")
study = run_hyperparameter_tuning(objective_catboost, N_TRIALS, *tuning_args)
print("Hyperparameter tuning finished.")

[I 2025-06-06 11:25:10,901] A new study created in memory with name: no-name-eb68848f-20bb-41fa-918e-c20075f84244


Starting hyperparameter tuning for...
0:	learn: 7720.3408014	test: 7637.6954852	best: 7637.6954852 (0)	total: 26.6ms	remaining: 1m 19s
1000:	learn: 1890.1042162	test: 1973.1559422	best: 1973.1559422 (1000)	total: 26.4s	remaining: 52.8s
2000:	learn: 1664.2410530	test: 1817.9764527	best: 1817.9764527 (2000)	total: 57.8s	remaining: 28.9s
2999:	learn: 1543.0166344	test: 1755.5982613	best: 1755.5703489 (2996)	total: 1m 25s	remaining: 0us

bestTest = 1755.570349
bestIteration = 2996

Shrink model to first 2997 iterations.


[I 2025-06-06 11:26:37,674] Trial 0 finished with value: 1755.5703489080192 and parameters: {'learning_rate': 0.05729304656391848, 'depth': 4, 'random_strength': 28.163816221843085, 'colsample_bylevel': 0.06583087546854849, 'min_data_in_leaf': 30, 'leaf_estimation_iterations': 4}. Best is trial 0 with value: 1755.5703489080192.


0:	learn: 7884.9853830	test: 7802.8181396	best: 7802.8181396 (0)	total: 81ms	remaining: 4m 2s
1000:	learn: 2758.5224949	test: 2767.5585247	best: 2767.5585247 (1000)	total: 1m 27s	remaining: 2m 54s
2000:	learn: 2422.7502887	test: 2452.9631138	best: 2452.9631138 (2000)	total: 2m 57s	remaining: 1m 28s
2999:	learn: 2048.5177964	test: 2089.7671903	best: 2089.7671903 (2999)	total: 4m 30s	remaining: 0us

bestTest = 2089.76719
bestIteration = 2999



[I 2025-06-06 11:31:09,107] Trial 1 finished with value: 2089.7671902700786 and parameters: {'learning_rate': 0.006301806884847529, 'depth': 5, 'random_strength': 22.677797600974753, 'colsample_bylevel': 0.575904383530925, 'min_data_in_leaf': 50, 'leaf_estimation_iterations': 15}. Best is trial 0 with value: 1755.5703489080192.


...

0:	learn: 7900.4625312	test: 7818.2715124	best: 7818.2715124 (0)	total: 71.9ms	remaining: 3m 35s
1000:	learn: 2953.5045671	test: 2952.9944937	best: 2952.9944937 (1000)	total: 2m 39s	remaining: 5m 18s
2000:	learn: 2581.4490582	test: 2627.7832222	best: 2627.7832222 (2000)	total: 5m 18s	remaining: 2m 39s
2999:	learn: 2364.9258330	test: 2434.7482092	best: 2434.7482092 (2999)	total: 7m 47s	remaining: 0us

bestTest = 2434.748209
bestIteration = 2999



[I 2025-06-06 11:59:20,386] Trial 8 finished with value: 2434.7482091690986 and parameters: {'learning_rate': 0.0038634981713881766, 'depth': 8, 'random_strength': 34.224432232821, 'colsample_bylevel': 0.9674591517188686, 'min_data_in_leaf': 5, 'leaf_estimation_iterations': 13}. Best is trial 2 with value: 1559.2665395709819.


0:	learn: 7906.6393500	test: 7824.4230858	best: 7824.4230858 (0)	total: 54.8ms	remaining: 2m 44s
1000:	learn: 3342.1024677	test: 3300.4785168	best: 3300.4785168 (1000)	total: 1m 51s	remaining: 3m 42s
2000:	learn: 2825.5157423	test: 2832.6608326	best: 2832.6608326 (2000)	total: 3m 21s	remaining: 1m 40s
2999:	learn: 2643.3787021	test: 2674.5527766	best: 2674.5527766 (2999)	total: 4m 41s	remaining: 0us

bestTest = 2674.552777
bestIteration = 2999



[I 2025-06-06 12:04:02,938] Trial 9 finished with value: 2674.5527765871393 and parameters: {'learning_rate': 0.0026749431978895424, 'depth': 7, 'random_strength': 34.363730780405994, 'colsample_bylevel': 0.29041710037902957, 'min_data_in_leaf': 25, 'leaf_estimation_iterations': 9}. Best is trial 2 with value: 1559.2665395709819.


Best trial number: 2
Best RMSE: 1559.2665
Best hyperparameters:
  learning_rate: 0.04008472597574163
  depth: 8
  random_strength: 49.48606173053552
  colsample_bylevel: 0.4322591692355058
  min_data_in_leaf: 37
  leaf_estimation_iterations: 13
Hyperparameter tuning finished.


In [156]:
new_categorical_variables = [
    "routeType",
    "RECATwake",
    "aircraftType",
    "airlineCode",
]

columns_to_remove = [
'CRUISE_modo_c_variance',
'DESCENT_vel_z_median',
'CLIMB_modo_c_median',
'DESCENT_distance',
'DESCENT_modo_c_median',
'DESCENT_vel_mod_median',
'CLIMB_vel_mod_median',
'CRUISE_vel_z_variance',
'DESCENT_duration',
'CRUISE_vel_z_median',
'numberOfEngines',
'DESCENT_vel_z_variance',
'CLIMB_delta_modo_c',
'CLIMB_vel_mod_variance',
'flightType',
    ]
new_cols = [col for col in X_train.columns if col not in columns_to_remove]
# Feature selection
X_train_new = X_train[new_cols]
X_test_new = X_test[new_cols]

In [157]:
best_params = study.best_trial.params.copy()
final_model = None
# iterations = study.best_trial.user_attrs.get("best_iteration", 1000)
cat_params = {
    **best_params,
    "iterations": 10000,
    "random_seed": 42,
    "loss_function": "RMSE",
    "task_type": "CPU",
    "allow_writing_files": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,    
}

final_model = CatBoostRegressor(**cat_params)

# train_pool = Pool(X_train, y_train, cat_features=CATEGORICAL_VARIABLES)
# test_pool = Pool(X_test, y_test, cat_features=CATEGORICAL_VARIABLES)
train_pool = Pool(X_train_new, y_train, cat_features=new_categorical_variables)
test_pool = Pool(X_test_new, y_test, cat_features=new_categorical_variables)
final_model.fit(
    train_pool,
    eval_set=test_pool,
    verbose=1000,
)

0:	learn: 7716.9807189	test: 7636.8983208	best: 7636.8983208 (0)	total: 119ms	remaining: 19m 45s
1000:	learn: 1333.5928297	test: 1651.9923958	best: 1651.9923958 (1000)	total: 2m 4s	remaining: 18m 37s
2000:	learn: 1017.3721931	test: 1586.7524744	best: 1586.7260096 (1996)	total: 4m 11s	remaining: 16m 44s
3000:	learn: 828.7416246	test: 1568.0539709	best: 1568.0204203 (2981)	total: 6m 23s	remaining: 14m 54s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1567.45211
bestIteration = 3108

Shrink model to first 3109 iterations.


In [158]:
y_pred = final_model.predict(X_test_new)
r2, mae, rmse, mape =  get_model_metrics(y_pred, X_test_new, y_test)    
print(f"Mean Absolute Error: {mae:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"R^2 Score: {r2:.4f}")

Mean Absolute Error: 1062.48
Root Mean Squared Error: 1567.45
R^2 Score: 0.9600


In [134]:
with open('CatBoost_M.pkl', 'wb') as file:  
    pickle.dump(final_model, file)

In [159]:
feature_importances = final_model.get_feature_importance(train_pool)
feature_names = X_train_new.columns
for score, name in sorted(zip(feature_importances, feature_names), reverse=True):
    print('{}: {}'.format(name, score))

aircraftType: 11.934925592376635
mfc: 7.494594255091928
RECATwake: 7.180169234663178
CRUISE_modo_c_median: 6.059403348816369
oew: 5.842971952405314
mlw: 5.124615092268049
Payload: 4.752890684708745
airlineCode: 4.541729777449332
cruiseLevel: 3.843261375697516
total_distance: 3.733355528127446
mtow_openap: 3.0621325365513212
mtow_rates: 2.9224326227243576
cruiseSpeed: 2.5642128000841677
ADEPLong: 2.35608374231325
ADESLong: 2.2697235804198326
seats: 1.9224034356553188
CLIMB_vel_z_median: 1.9013572467968722
flight_duration: 1.8635481729868453
CRUISE_distance: 1.7478649319598345
CRUISE_duration: 1.5620393765346436
ATOT_track_day: 1.5091437300843615
ALDT_track_day: 1.4475416811125914
CLIMB_duration: 1.4121240919601998
ADEPLat: 1.401308712951665
DESCENT_delta_modo_c: 1.371653459834242
ADESLat: 1.2569794934725564
CLIMB_modo_c_variance: 1.138192838298098
routeType: 1.1369084038850321
DESCENT_modo_c_variance: 0.9822508511671384
CLIMB_vel_z_variance: 0.8217298249968378
CLIMB_distance: 0.77617539

In [160]:
for score, name in sorted(zip(feature_importances, feature_names), reverse=True):
    if score < 0.5:
        print("'{}',".format(name))

In [161]:
feature_names

Index(['ADEPLat', 'ADEPLong', 'ADESLat', 'ADESLong', 'routeType',
       'aircraftType', 'airlineCode', 'RECATwake', 'cruiseSpeed',
       'cruiseLevel', 'ATOT_track_hour', 'ATOT_track_day', 'ALDT_track_hour',
       'ALDT_track_day', 'flight_duration', 'CLIMB_duration',
       'CRUISE_duration', 'total_distance', 'CLIMB_distance',
       'CRUISE_distance', 'CLIMB_modo_c_variance', 'CLIMB_vel_z_median',
       'CLIMB_vel_z_variance', 'CRUISE_modo_c_median', 'CRUISE_vel_mod_median',
       'CRUISE_vel_mod_variance', 'CRUISE_delta_modo_c',
       'DESCENT_modo_c_variance', 'DESCENT_vel_mod_variance',
       'DESCENT_delta_modo_c', 'mtow_openap', 'mlw', 'mfc', 'oew',
       'mtow_rates', 'seats', 'Payload'],
      dtype='object')